# load_* 결과 확인용

합의한 A3 전처리가 모두 적용됐는지 원본과 비교해 확인합니다.

In [2]:
import pandas as pd
from bookrec.data import load_book_tags, load_books, load_ratings, load_to_read
from bookrec.data.loaders import RAW_DIR

pd.set_option("display.max_colwidth", 70)
raw = {name: pd.read_csv(RAW_DIR / f"{name}.csv") for name in ["ratings", "to_read", "books", "book_tags"]}
ratings, to_read, books, book_tags = load_ratings(), load_to_read(), load_books(), load_book_tags()

checks = []
def check(name, ok):
    checks.append((name, "OK" if ok else "FAIL"))

## load_ratings()

In [4]:
check("ratings: 원본과 행 순서·값이 완전히 같음", ratings.equals(raw["ratings"]))
print(ratings.shape); ratings.head()

(5976479, 3)


,user_id,book_id,rating
0,1,258,5
1,2,4081,4
2,2,260,5
3,2,9296,5
4,2,2318,3


## load_to_read()

In [6]:
check("to_read: 원본과 완전히 같음", to_read.equals(raw["to_read"]))
print(to_read.shape); to_read.head(50)

(912705, 2)


,user_id,book_id
0,9,8
1,15,398
2,15,275
3,37,7173
4,34,380
5,34,483
6,34,8598
7,34,3581
8,70,498
9,76,4250


## load_books()

In [8]:
check("books: goodreads_book_id만 빠지고 나머지 원본 컬럼은 모두 있음", set(raw["books"].columns) - set(books.columns) == {"goodreads_book_id"})
check("books: authors는 목록, main_author는 첫 저자", (books["authors"].str[0] == books["main_author"]).all() and books["authors"].map(type).eq(list).all())
check("books: 저자 이름에 연속 공백·앞뒤 공백 없음", books["authors"].explode().map(lambda n: n == " ".join(n.split())).all())
check("books: title 앞뒤 공백 없음", (books["title"] == books["title"].str.strip()).all())
check("books: 출판연도 결측 21권·기원전 31권 그대로", books["original_publication_year"].isna().sum() == 21 and (books["original_publication_year"] <= 0).sum() == 31)
check("books: language_code 결측 1,084권 그대로", books["language_code"].isna().sum() == 1084)
print(books.shape)
books[["book_id", "title", "authors", "main_author", "original_publication_year", "language_code"]].head(50)

(10000, 23)


,book_id,title,authors,main_author,original_publication_year,language_code
0,1,"The Hunger Games (The Hunger Games, #1)",[Suzanne Collins],Suzanne Collins,2008.0,eng
1,2,"Harry Potter and the Sorcerer's Stone (Harry Potter, #1)","[J.K. Rowling, Mary GrandPré]",J.K. Rowling,1997.0,eng
2,3,"Twilight (Twilight, #1)",[Stephenie Meyer],Stephenie Meyer,2005.0,en-US
3,4,To Kill a Mockingbird,[Harper Lee],Harper Lee,1960.0,eng
4,5,The Great Gatsby,[F. Scott Fitzgerald],F. Scott Fitzgerald,1925.0,eng
5,6,The Fault in Our Stars,[John Green],John Green,2012.0,eng
6,7,The Hobbit,[J.R.R. Tolkien],J.R.R. Tolkien,1937.0,en-US
7,8,The Catcher in the Rye,[J.D. Salinger],J.D. Salinger,1951.0,eng
8,9,"Angels & Demons (Robert Langdon, #1)",[Dan Brown],Dan Brown,2000.0,en-CA
9,10,Pride and Prejudice,[Jane Austen],Jane Austen,1813.0,eng


## load_book_tags()

In [10]:
check("book_tags: 컬럼은 book_id, tag_name, count", list(book_tags.columns) == ["book_id", "tag_name", "count"])
check("book_tags: 음수 count 없음 (6행 삭제)", (book_tags["count"] >= 0).all())
check("book_tags: 중복 (책, 태그) 없음, 8행 줄어듦", not book_tags.duplicated(["book_id", "tag_name"]).any() and len(raw["book_tags"]) - len(book_tags) == 14)
check("book_tags: 해리 포터 owned는 큰 값 6066 하나", book_tags.query("book_id == 2 and tag_name == 'owned'")["count"].tolist() == [6066])
check("book_tags: 1만 권 모두 태그 있음", book_tags["book_id"].nunique() == 10000)
print(book_tags.shape)
book_tags.query("book_id == 2").head(50)

(999898, 3)


,book_id,tag_name,count
100,2,to-read,496107
101,2,favorites,48220
102,2,fantasy,47478
103,2,currently-reading,44640
104,2,young-adult,14984
105,2,fiction,13239
106,2,harry-potter,8559
107,2,books-i-own,7917
108,2,owned,6066
109,2,ya,5276


## 저장 파일이 생기지 않았는지

In [12]:
check("data/에는 원본 CSV 5개만 있음 (정리한 파일을 저장하지 않음)", sorted(p.name for p in RAW_DIR.parent.rglob("*") if p.is_file()) == sorted(["books.csv", "book_tags.csv", "tags.csv", "ratings.csv", "to_read.csv"]))
pd.DataFrame(checks, columns=["합의 사항", "결과"])

,합의 사항,결과
0,ratings: 원본과 행 순서·값이 완전히 같음,OK
1,ratings: 원본과 행 순서·값이 완전히 같음,OK
2,to_read: 원본과 완전히 같음,OK
3,to_read: 원본과 완전히 같음,OK
4,books: goodreads_book_id만 빠지고 나머지 원본 컬럼은 모두 있음,OK
5,"books: authors는 목록, main_author는 첫 저자",OK
6,books: 저자 이름에 연속 공백·앞뒤 공백 없음,OK
7,books: title 앞뒤 공백 없음,OK
8,books: 출판연도 결측 21권·기원전 31권 그대로,OK
9,"books: language_code 결측 1,084권 그대로",OK
